In [1]:
from __future__ import print_function,division
from builtins import range

In [2]:
import numpy as np 

In [3]:
SMALL_ENOUGH =1e-3
GAMMA=.9

In [4]:
class WindyGrid:
    def __init__(self,rows,cols,start):
        self.rows=rows
        self.cols=cols
        self.i=start[0]
        self.j=start[1]
    
    def set(self,rewards,actions,probs):
        self.rewards=rewards
        self.actions=actions
        self.probs=probs
    
    def set_state(self,s):
        self.i=s[0]
        self.j=s[1]
    
    def current_state(self):
        return (self.i,self.j)

    def is_terminal(self,s):
        return s not in self.actions

    def move (self,action):
        s=(self.i,self.j)
        a=action
        next_state_probs=self.probs[(s,a)]
        next_state=list(next_state_porbs.keys())
        next_probs=list(next_state_porbs.values())
        s2=np.random.choice(next_state,p=next_probs)
        self.i,self.j=s2 
        return self.reward.get(s2,0)

    def undo_move (self,action):
        if action =='U':
            self.i +=1
        if action =='D':
            self.i -=1
        if action =='R':
            self.j -=1
        if action =='L':
            self.j +=1
        assert(self.current_state() in self.all_stats())

    def game_over (self):
        return (self.i,self.j) not in self.actions

    def all_states(self):
        return set(self.actions.keys()) | set (self.rewards.keys())


In [5]:
def Windy_grid_penalized(step_cost=0):
    g=WindyGrid(3,4,(2,0))
    rewards={
        (0,0):step_cost,
        (0,1):step_cost,
        (0,2):step_cost,
        (1,0):step_cost,
        (1,2):step_cost,
        (2,0):step_cost,
        (2,1):step_cost,
        (2,2):step_cost,
        (2,3):step_cost,
        (0,3):1,
        (1,3):-1
        
    }

    actions={
        (0,0):('D','R'),
        (0,1):('L','R'),
        (0,2):('L','R','D'),
        (1,0):('U','D'),
        (1,2):('R','U','D'),
        (2,0):('U','R'),
        (2,1):('L','R'),
        (2,2):('L','R','U'),
        (2,3):('L','U')
    }
    probs={
        ((0,0),'R'):{(0,1):1},
        ((0,0),'D'):{(1,0):1},
        
        ((0,1),'R'):{(0,2):1},
        ((0,1),'L'):{(0,0):1},
        
        ((0,2),'R'):{(0,3):1},
        ((0,2),'L:'):{(0,1):1},
        ((0,2),'D'):{(1,2):1},
        
        ((1,0),'D'):{(2,0):1},
        ((1,0),'U'):{(0,0):1},
        
        ((1,2),'U'):{(0,2):1},
        ((1,2),'R'):{(1,3):1},
        ((1,2),'D'):{(2,2):1},
        
        ((2,0),'R'):{(2,1):1},
        ((2,0),'U'):{(1,0):1},
        
        ((2,1),'R'):{(2,2):1},
        ((2,1),'L'):{(2,0):1},
        
        ((2,2),'R'):{(2,3):1},
        ((2,2),'L'):{(2,1):1},
        ((2,2),'U'):{(1,2):1},
        
        ((2,3),'L'):{(2,2):1},
        ((2,3),'U'):{(1,3):1},
    }
    
    g.set(rewards,actions,probs)
    return g
    

In [6]:
def print_values(V,g):
    for i in range (g.rows):
        print("--------------------------")
        for j in range(g.cols):
            v=V.get((i,j),0)
            if v >=0:
                print(" %.2f |"% v,end="")
            else:
                print("%.2f |" %v,end="")
        print("")
        
        
def print_policy(P,g):
    for i in range(g.rows):
        print("----------------------------")
        for j in range(g.cols):
            a=P.get((i,j),' ')
            print(" %s |" %a,end="")
        print("")
    


In [7]:
def get_transition_probs_and_rewards(grid):
    transition_prob={}
    rewards={}
    
    for (s,a),v in grid.probs.items():
        for s2,p in v.items():
            transition_prob[(s,a,s2)]=p
            rewards[(s,a,s2)]=grid.rewards.get(s2,0)
                         
    return transition_prob, rewards

In [10]:
if __name__=='__main__':
    grid=Windy_grid_penalized(0)
    #ACTION_SPACE=['U','D','L','R']
    transition_prob,rewards=get_transition_probs_and_rewards(grid)

    
    print("rewards:")
    print_values(grid.rewards,grid)
    
    V={}
    states=grid.all_states()
    #ACTION_SPACE=['U','D','L','R']
    for s in states:
        V[s]=0
        
    it=0
    while True:
        biggest_change=0
        for s in grid.all_states():
    
            if not grid.is_terminal(s):
                old_v=V[s]
                new_v=float('-inf')
                
                for a in grid.actions[s]:
                    v=0
                    for s2 in grid.all_states():
                        #action_prob=1 if policy.get(s)==a else 0
                        r=rewards.get((s,a,s2),0)
                        v +=transition_prob.get((s,a,s2),0) *(r+GAMMA*V[s2])
                    if v>new_v:
                        new_v=v
                        
                V[s]=new_v
                biggest_change = max(biggest_change,np.abs(old_v-V[s]))
                    
        it +=1
        if biggest_change < SMALL_ENOUGH:
            break
        
    policy={}
    for s in grid.actions.keys():
        best_a=None 
        best_value=float('-inf')
        for a in grid.actions[s]:
            v=0
            for s2 in grid.all_states():
                r=rewards.get((s,a,s2),0)
                v +=transition_prob.get((s,a,s2),0) *(r+GAMMA*V[s2])
        
            if v>best_value:
                best_value=v 
                best_a=a 
        policy[s]=best_a
        
    print("Values:")
    print_values(V,grid)
    print("policy:")
    print_policy(policy,grid)
   

rewards:
--------------------------
 0.00 | 0.00 | 0.00 | 1.00 |
--------------------------
 0.00 | 0.00 | 0.00 |-1.00 |
--------------------------
 0.00 | 0.00 | 0.00 | 0.00 |
Values:
--------------------------
 0.81 | 0.90 | 1.00 | 0.00 |
--------------------------
 0.73 | 0.00 | 0.90 | 0.00 |
--------------------------
 0.66 | 0.73 | 0.81 | 0.73 |
policy:
----------------------------
 R | R | R |   |
----------------------------
 U |   | U |   |
----------------------------
 U | R | U | L |
